# Create Azure Container Apps GPU Job for GPT Pretraining

This notebook creates a GPU-enabled Azure Container Apps Environment and a manual Azure Container Apps Job for the public Docker Hub image `docker.io/deril2605/gpt-pretraining-smoke:latest`.

It is modeled after your earlier Container Apps GPU notebook, but targets a **job** instead of a long-running app and defaults to the largest serverless GPU workload profile shown in your screenshot: `Consumption-GPU-NC24-A100`.

This version also explicitly creates or reuses a **Log Analytics Workspace** and attaches it to the Container Apps Environment so Azure Portal log links are enabled.

Before running:
- Make sure your subscription has Container Apps GPU quota for A100 in the selected region.
- Make sure the resource provider `Microsoft.App` is registered.
- Make sure your local environment can authenticate with Azure via `InteractiveBrowserCredential` in the correct tenant.

Useful Microsoft docs:
- Jobs overview: https://learn.microsoft.com/en-us/azure/container-apps/jobs
- Workload profiles: https://learn.microsoft.com/en-us/azure/container-apps/workload-profiles-overview
- GPU serverless overview: https://learn.microsoft.com/en-us/azure/container-apps/gpu-serverless-overview
- ARM schema for jobs: https://learn.microsoft.com/en-us/azure/templates/microsoft.app/2025-01-01/jobs


In [ ]:
%pip install azure-identity azure-mgmt-appcontainers azure-mgmt-resource python-dotenv requests


In [14]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from azure.identity import InteractiveBrowserCredential
from azure.mgmt.appcontainers import ContainerAppsAPIClient
from azure.mgmt.loganalytics import LogAnalyticsManagementClient
from azure.mgmt.resource import ResourceManagementClient

load_dotenv()
load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd() / 'aca_pretraining' / '.env')

SUBSCRIPTION_ID = os.environ['AZURE_SUBSCRIPTION_ID']
AZURE_TENANT_ID = os.environ['AZURE_TENANT_ID']
RESOURCE_GROUP = os.environ['AZURE_RESOURCE_GROUP']
LOCATION = os.getenv('AZURE_LOCATION', 'eastus2')

ENV_NAME = os.getenv('AZURE_CONTAINERAPPS_ENV_NAME', 'cae-gpt-pretraining-gpu')
JOB_NAME = os.getenv('AZURE_CONTAINERAPP_JOB_NAME', 'gpt-pretraining-job')
GPU_PROFILE_NAME = os.getenv('AZURE_GPU_PROFILE_NAME', 'Consumption-GPU-NC24-A100')

LOG_ANALYTICS_WORKSPACE_NAME = os.getenv('AZURE_LOG_ANALYTICS_WORKSPACE_NAME', 'law-gpt-pretraining')
LOG_ANALYTICS_SKU = os.getenv('AZURE_LOG_ANALYTICS_SKU', 'PerGB2018')
LOG_ANALYTICS_RETENTION_DAYS = int(os.getenv('AZURE_LOG_ANALYTICS_RETENTION_DAYS', '30'))

TRAINING_IMAGE = os.getenv('ACA_JOB_IMAGE', 'docker.io/deril2605/gpt-pretraining-smoke:latest')
CPU = float(os.getenv('ACA_JOB_CPU', '24'))
MEMORY = os.getenv('ACA_JOB_MEMORY', '220Gi')
REPLICA_TIMEOUT_SECONDS = int(os.getenv('ACA_JOB_REPLICA_TIMEOUT_SECONDS', '86400'))
REPLICA_RETRY_LIMIT = int(os.getenv('ACA_JOB_REPLICA_RETRY_LIMIT', '0'))
PARALLELISM = int(os.getenv('ACA_JOB_PARALLELISM', '1'))
REPLICA_COMPLETION_COUNT = int(os.getenv('ACA_JOB_REPLICA_COMPLETION_COUNT', '1'))

BLOB_CONNECTION_STRING = os.environ['AZURE_STORAGE_CONNECTION_STRING']
BLOB_CONTAINER = os.environ['AZURE_BLOB_CONTAINER']

ARM_API_VERSION = '2025-01-01'

print(json.dumps({
    'subscription_id': SUBSCRIPTION_ID,
    'tenant_id': AZURE_TENANT_ID,
    'resource_group': RESOURCE_GROUP,
    'location': LOCATION,
    'environment_name': ENV_NAME,
    'job_name': JOB_NAME,
    'gpu_profile_name': GPU_PROFILE_NAME,
    'log_analytics_workspace_name': LOG_ANALYTICS_WORKSPACE_NAME,
    'training_image': TRAINING_IMAGE,
    'cpu': CPU,
    'memory': MEMORY,
}, indent=2))


{
  "subscription_id": "67f419e5-9aac-4976-ba8c-b2ac9f3895f3",
  "tenant_id": "18c4eca2-960b-4691-8102-fcf828b19842",
  "resource_group": "eus2rgidpd01",
  "location": "eastus",
  "environment_name": "cae-gpt-pretraining-gpu",
  "job_name": "gpt-pretraining-job",
  "gpu_profile_name": "Consumption-GPU-NC24-A100",
  "log_analytics_workspace_name": "law-gpt-pretraining",
  "training_image": "docker.io/deril2605/gpt-pretraining-smoke:latest",
  "cpu": 24.0,
  "memory": "220Gi"
}


In [15]:
credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
app_client = ContainerAppsAPIClient(credential=credential, subscription_id=SUBSCRIPTION_ID)
log_analytics_client = LogAnalyticsManagementClient(credential, SUBSCRIPTION_ID)

resource_client.resource_groups.create_or_update(RESOURCE_GROUP, {'location': LOCATION})
print(f'Resource group ready: {RESOURCE_GROUP}')


Resource group ready: eus2rgidpd01


## Create or reuse Log Analytics

This ensures the Container Apps Environment has explicit Portal log support.


In [7]:
workspace_async = log_analytics_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP,
    LOG_ANALYTICS_WORKSPACE_NAME,
    {
        'location': LOCATION,
        'sku': {'name': LOG_ANALYTICS_SKU},
        'retention_in_days': LOG_ANALYTICS_RETENTION_DAYS,
    },
)
workspace = workspace_async.result()
workspace_keys = log_analytics_client.shared_keys.get_shared_keys(RESOURCE_GROUP, LOG_ANALYTICS_WORKSPACE_NAME)
LOG_ANALYTICS_CUSTOMER_ID = workspace.customer_id
LOG_ANALYTICS_SHARED_KEY = workspace_keys.primary_shared_key

print('Log Analytics workspace id:', workspace.id)
print('Log Analytics customer id:', LOG_ANALYTICS_CUSTOMER_ID)


Log Analytics workspace id: /subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.OperationalInsights/workspaces/law-gpt-pretraining
Log Analytics customer id: ad24b6d9-d7fd-4737-bd40-9b71c4c88b29


In [8]:
env_def = {
    'location': LOCATION,
    'properties': {
        'appLogsConfiguration': {
            'destination': 'log-analytics',
            'logAnalyticsConfiguration': {
                'customerId': LOG_ANALYTICS_CUSTOMER_ID,
                'sharedKey': LOG_ANALYTICS_SHARED_KEY,
            },
        },
        'workloadProfiles': [
            {'name': 'Consumption', 'workloadProfileType': 'Consumption'},
            {'name': GPU_PROFILE_NAME, 'workloadProfileType': GPU_PROFILE_NAME},
        ],
    },
}

env = app_client.managed_environments.begin_create_or_update(
    resource_group_name=RESOURCE_GROUP,
    environment_name=ENV_NAME,
    environment_envelope=env_def,
).result()

ENV_RESOURCE_ID = env.id
print('Managed environment ready:', ENV_RESOURCE_ID)


Managed environment ready: /subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/managedEnvironments/cae-gpt-pretraining-gpu


In [9]:
env = app_client.managed_environments.get(RESOURCE_GROUP, ENV_NAME)

print('Managed Environment id:', getattr(env, 'id', None))
print('Location:', getattr(env, 'location', None))

wps = None
if hasattr(env, 'workload_profiles'):
    wps = env.workload_profiles
elif hasattr(env, 'properties') and hasattr(env.properties, 'workload_profiles'):
    wps = env.properties.workload_profiles
else:
    wps = []

print('Workload profiles on environment:')
for wp in (wps or []):
    name = getattr(wp, 'name', None)
    wptype = getattr(wp, 'workload_profile_type', None) or getattr(wp, 'workloadProfileType', None)
    print(' - name:', name, '| type:', wptype)


Managed Environment id: /subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/managedEnvironments/cae-gpt-pretraining-gpu
Location: East US
Workload profiles on environment:
 - name: Consumption | type: Consumption
 - name: Consumption-GPU-NC24-A100 | type: Consumption-GPU-NC24-A100


## Create the ACA Job

This job uses your public Docker Hub image, so no registry credentials are needed. Blob credentials are passed in as a Container Apps secret and referenced from the container environment.


In [16]:
job_payload = {
    'location': LOCATION,
    'properties': {
        'environmentId': ENV_RESOURCE_ID,
        'workloadProfileName': GPU_PROFILE_NAME,
        'configuration': {
            'triggerType': 'Manual',
            'replicaTimeout': REPLICA_TIMEOUT_SECONDS,
            'replicaRetryLimit': REPLICA_RETRY_LIMIT,
            'manualTriggerConfig': {
                'parallelism': PARALLELISM,
                'replicaCompletionCount': REPLICA_COMPLETION_COUNT,
            },
            'secrets': [
                {
                    'name': 'azure-storage-connection-string',
                    'value': BLOB_CONNECTION_STRING,
                }
            ],
        },
        'template': {
            'containers': [
                {
                    'name': 'trainer',
                    'image': TRAINING_IMAGE,
                    'resources': {
                        'cpu': CPU,
                        'memory': MEMORY,
                    },
                    'env': [
                        {'name': 'AZURE_BLOB_CONTAINER', 'value': BLOB_CONTAINER},
                        {'name': 'AZURE_STORAGE_CONNECTION_STRING', 'secretRef': 'azure-storage-connection-string'},
                        {'name': 'PYTHONUNBUFFERED', 'value': '1'},
                    ],
                }
            ]
        },
    },
}

job_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}'
    f'?api-version={ARM_API_VERSION}'
)

token = credential.get_token('https://management.azure.com/.default').token
headers = {
    'Authorization': f'Bearer {token}',
    'Content-Type': 'application/json',
}

response = requests.put(job_url, headers=headers, json=job_payload, timeout=1800)
response.raise_for_status()
job_response = response.json()
print(json.dumps(job_response, indent=2)[:5000])


{
  "id": "/subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/jobs/gpt-pretraining-job",
  "name": "gpt-pretraining-job",
  "type": "Microsoft.App/jobs",
  "location": "East US",
  "systemData": {
    "createdBy": "nomuladheeraj@gmail.com",
    "createdByType": "User",
    "createdAt": "2026-03-27T06:27:38.1098642",
    "lastModifiedBy": "nomuladheeraj@gmail.com",
    "lastModifiedByType": "User",
    "lastModifiedAt": "2026-03-27T06:52:11.1100428Z"
  },
  "properties": {
    "provisioningState": "InProgress",
    "environmentId": "/subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/managedEnvironments/cae-gpt-pretraining-gpu",
    "workloadProfileName": "Consumption-GPU-NC24-A100",
    "outboundIpAddresses": [
      "20.62.237.242"
    ],
    "configuration": {
      "secrets": [
        {
          "name": "azure-storage-connection-string"
        }
      ],
      "triggerType": "

In [17]:
job_get = requests.get(job_url, headers=headers, timeout=300)
job_get.raise_for_status()
job = job_get.json()

print('Job id:', job.get('id'))
print('Provisioning state:', job.get('properties', {}).get('provisioningState'))
print('Workload profile:', job.get('properties', {}).get('workloadProfileName'))
print('Image:', job.get('properties', {}).get('template', {}).get('containers', [{}])[0].get('image'))


Job id: /subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/jobs/gpt-pretraining-job
Provisioning state: InProgress
Workload profile: Consumption-GPU-NC24-A100
Image: docker.io/deril2605/gpt-pretraining-smoke:latest


## Start a manual execution

This sends the documented ARM `start` request for a Container Apps job.


In [18]:
start_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}/start'
    f'?api-version=2024-03-01'
)

start_response = requests.post(start_url, headers=headers, timeout=1800)
start_response.raise_for_status()
execution_response = start_response.json()
print(json.dumps(execution_response, indent=2)[:5000])


{
  "id": "/subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/jobs/gpt-pretraining-job/executions/gpt-pretraining-job-g6o0ht2",
  "name": "gpt-pretraining-job-g6o0ht2"
}


In [13]:
portal_job_url = (
    'https://portal.azure.com/#view/HubsExtension/BrowseResource/resourceType/Microsoft.App%2Fjobs'
)
resource_url = (
    f'https://portal.azure.com/#@/resource/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}/overview'
)

print('Jobs blade:', portal_job_url)
print('This job:', resource_url)


Jobs blade: https://portal.azure.com/#view/HubsExtension/BrowseResource/resourceType/Microsoft.App%2Fjobs
This job: https://portal.azure.com/#@/resource/subscriptions/67f419e5-9aac-4976-ba8c-b2ac9f3895f3/resourceGroups/eus2rgidpd01/providers/Microsoft.App/jobs/gpt-pretraining-job/overview


## Environment variables needed

Put these in your repo-root `.env` or `aca_pretraining/.env` before running the notebook:

- `AZURE_SUBSCRIPTION_ID`
- `AZURE_TENANT_ID`
- `AZURE_RESOURCE_GROUP`
- `AZURE_LOCATION`
- `AZURE_CONTAINERAPPS_ENV_NAME`
- `AZURE_CONTAINERAPP_JOB_NAME`
- `AZURE_GPU_PROFILE_NAME`
- `AZURE_LOG_ANALYTICS_WORKSPACE_NAME`
- `ACA_JOB_IMAGE`
- `ACA_JOB_CPU`
- `ACA_JOB_MEMORY`
- `AZURE_STORAGE_CONNECTION_STRING`
- `AZURE_BLOB_CONTAINER`

Recommended defaults for your current setup:

- `AZURE_GPU_PROFILE_NAME=Consumption-GPU-NC24-A100`
- `AZURE_LOG_ANALYTICS_WORKSPACE_NAME=law-gpt-pretraining`
- `ACA_JOB_IMAGE=docker.io/deril2605/gpt-pretraining-smoke:latest`
- `ACA_JOB_CPU=24`
- `ACA_JOB_MEMORY=220Gi`
